# Connect to Device

In [1]:
import pyvisa
import numpy as np
import pandas as pd
from pathlib import Path
from time import sleep
import matplotlib.pyplot as plt

rm = pyvisa.ResourceManager()
print(rm.list_resources())

# Change to match your instrument
sa: pyvisa.resources.USBInstrument = rm.open_resource("USB0::0xF4EC::0x1301::SSA3XNEC6R0076::INSTR")
sa.timeout = 30000
print(sa.query("*IDN?"))



ValueError: Could not locate a VISA implementation. Install either the IVI binary or pyvisa-py.

# Set parameters

In [ ]:
# ============================================================
# USER SETTINGS
# ============================================================
def make_freq_array(start_freq, stop_freq): #Creates array of successive frequency ranges in the frequency list. --> [a,b,c] --> [[a,b],[b,c]]
    current_freq = start_freq
    freq_array = []
    while 10*current_freq < stop_freq:
        freq_array.append([current_freq, current_freq*10])
        current_freq *=10

    freq_array.append([current_freq, stop_freq])
    
    return freq_array

def make_measurement_arrays(freq_array, rbw_span_ratio, vbw_rbw_ratio):
    spans=[]
    centers=[]
    rbws=[]
    vbws=[]
    for start_freq, end_freq in freq_array:
        span = end_freq - start_freq
        center = (end_freq+start_freq)/2
        rbw = span * rbw_span_ratio
        vbw = rbw * vbw_rbw_ratio
        spans.append(span)
        centers.append(center)
        rbws.append(rbw)
        vbws.append(vbw)
    return spans, centers, rbws, vbws

def prep_for_measurement():
    sa.write(":INIT:CONT OFF") # Turn off continuous acquisition
    sa.write(":DET NORMAL")

def collect_data(spans, centers, rbws, vbws):
    full_freqs = np.array([])
    full_trace = np.array([])
    full_rbws = np.array([])
    full_vbws = np.array([])
    for span, center, rbw, vbw in zip(spans, centers, rbws, vbws):
        # Configure
        sa.write(f":FREQ:CENT {center}")
        sa.write(f":FREQ:SPAN {span}")
        sa.write(f":BWID {rbw}")
        sa.write(f":BWID:VID {vbw}")

        # Allow settling
        sleep(1)

        sa.write(":INIT")
        sa.query("*OPC?")

        start_freq = float(sa.query(":FREQ:STAR?"))
        stop_freq  = float(sa.query(":FREQ:STOP?"))
        vbw_current = float(sa.query(":BWID:VID?"))
        rbw_current = float(sa.query(":BWID?"))
        trace = sa.query(":TRAC:DATA? TRACE1")
        trace = np.array(trace.split(","), dtype=float)

        freqs = np.linspace(
            start_freq,
            stop_freq,
            len(trace)
        )
        full_freqs = np.concatenate((full_freqs, freqs))
        full_trace = np.concatenate((full_trace, trace))
        full_rbws = np.concatenate((full_rbws, rbw_current))
        full_vbws = np.concatenate((full_rbws, vbw_current))

    df = pd.DataFrame({
        "Frequency (Hz)": full_freqs,
        "Power (dBm)": full_trace,
        "RBWs (Hz)": full_rbws,
        "VBWs (Hz)": full_vbws,
    })
    return df

def plot_data(data: pd.DataFrame):
    plt.semilogx(data['Frequency (Hz)'], data['Power (dBm)'])

def save_data(path: Path, data: pd.DataFrame):
    save_dir = path.parent
    save_dir.mkdir(exist_ok=True)
    data.to_csv(path, index=False)

# Collect data

In [ ]:
start_freq = 10e3
stop_freq = 100e6
rbw_span_ratio = .001
vbw_rbw_ratio = .1

freq_array = make_freq_array(start_freq, stop_freq)
spans, centers, rbws, vbws = make_measurement_arrays(freq_array, rbw_span_ratio, vbw_rbw_ratio)
data = collect_data(spans, centers, rbws, vbws)
plot_data(data)